# Launch: western states (campaign 5)

Everything below is filled in against real, provisioned infrastructure. It is
[5_submit_job_parquet.ipynb](./5_submit_job_parquet.ipynb) with the values
already resolved, so a launch is a matter of reading and running rather than
configuring.

Conventions and the reasoning behind them:
[17_launch_conventions.md](../docs/rerun_2026/17_launch_conventions.md).

## ⚠️ Two decisions to confirm before running section 4

**1. The weight changed from what 09 specified.** This campaign is a
stakeholder deliverable. [09_western_states_run.md](../docs/rerun_2026/09_western_states_run.md)
specifies `instance`; [11_launch_plan.md](../docs/rerun_2026/11_launch_plan.md)
supersedes it with `original`, because `instance` has a ceiling on dense
near-field aftershocks — at Ridgecrest it emits 246 S picks with its threshold
on the floor where the others reach 684 and 832. The substitution is defensible
but **changes what the stakeholder receives**, so it should be their call, not
ours.

**2. This is the largest campaign, not the smallest.** 33.8M station-days, 65%
of the launch total. v3 has so far run four shards end to end. Section 5 is a
smoke test that stops after two shards; the case for running a smaller campaign
first (SCEDC, 2.47M station-days) is real.

## Provisioned and verified

| | |
|---|---|
| Bucket | `s3://quakescope-picks-2026` (us-east-2, versioning off, public access blocked) |
| Campaign prefix | `s3://quakescope-picks-2026/western` |
| Job queue | `niyiyu_earthscope_missing_station` |
| Compute env | `niyiyu_earthscope` — FARGATE_SPOT, maxvCpus **4000** |
| Job definition | `quakescope_v3_worker:2`, image pinned to `9abd01c` |
| Role | `SeisBenchBatchRole` (job + execution), S3 write verified by simulation |
| Stations | `sb_catalog/configs/networks/western_states.csv` — 24,111 across 122 networks |

In [ ]:
import sys, json, time

sys.path.append("..")

import boto3
import pandas as pd

from sb_catalog.src.s3_state import S3CampaignState
from sb_catalog.src.shard_planner import plan, parse_year_day

REGION      = "us-east-2"
CAMPAIGN    = "s3://quakescope-picks-2026/western"
JOB_QUEUE   = "niyiyu_earthscope_missing_station"
JOB_DEF     = "quakescope_v3_worker"      # rev 2, image pinned to 9abd01c
WEIGHT      = "original"                  # see decision 1 above
START, END  = "2010.001", "2026.001"

batch = boto3.client("batch", region_name=REGION)
state = S3CampaignState(CAMPAIGN)
print("campaign:", CAMPAIGN)

## 1. Preflight

Checks the things that otherwise fail as thousands of tasks rather than as one
readable error: that the image can run the worker, that the queue and compute
environment are enabled, and that the job definition does not carry
`--classifier`.

In [ ]:
ok = True

jd = [d for d in batch.describe_job_definitions(jobDefinitionName=JOB_DEF,
                                                status="ACTIVE")["jobDefinitions"]]
jd = max(jd, key=lambda d: d["revision"])
cp = jd["containerProperties"]
print(f"job definition : {JOB_DEF}:{jd['revision']}")
print(f"  image        : {cp['image']}")
if cp["image"].endswith(":latest"):
    print("  !! :latest moves under a running campaign - pin a short-SHA tag")
    ok = False
if "--classifier" in cp["command"]:
    print("  !! --classifier is hardcoded; it is out for 2026")
    ok = False

q = batch.describe_job_queues(jobQueues=[JOB_QUEUE])["jobQueues"][0]
ce_arn = q["computeEnvironmentOrder"][0]["computeEnvironment"]
ce = batch.describe_compute_environments(computeEnvironments=[ce_arn])["computeEnvironments"][0]
print(f"queue          : {q['state']}/{q['status']}")
print(f"compute env    : {ce['computeResources']['type']}, maxvCpus {ce['computeResources']['maxvCpus']}")
ok &= q["state"] == "ENABLED" and ce["state"] == "ENABLED"

print(f"\npreflight {'PASSED' if ok else 'FAILED - fix the above first'}")

## 2. Station metadata

Selected by **true state polygons**, not a bounding box. A single box over the
six states sweeps in about 3,000 stations from AZ, UT, MT and CO, which
[09](../docs/rerun_2026/09_western_states_run.md) explicitly warns against.
Nevada is the tell: under a naive per-state rectangle scheme it gets 3 stations,
because California's rectangle covers it.

In [ ]:
stations = pd.read_csv("../sb_catalog/configs/networks/western_states.csv")
stations["location_code"] = stations.location_code.fillna("").astype(str)
print(f"{len(stations):,} stations, {stations.network_code.nunique()} networks")
print(stations.state.value_counts().to_string())

state.write_stations(stations)

## 3. Plan the queue

**Immutable once written.** Completed work is keyed on shard id, so changing the
date range later means a new campaign prefix, not a rewritten queue. Read the
numbers before running the write cell.

In [ ]:
sel = state.get_stations()
shards = plan(sel, parse_year_day(START), parse_year_day(END))
sd = sum(s["n_station_days"] for s in shards)

print(f"{len(sel):,} stations x {START}..{END}")
print(f"  shards       : {len(shards):,}")
print(f"  station-days : {sd:,}")
print(f"  at ~34 s/band-day -> {sd*34/3600:,.0f} vCPU-hours")
print(f"  at $0.0148/vCPU-hr -> ~${sd*34/3600*0.0148:,.0f}")
print("\n  NOTE: processes per vCPU is unmeasured and swings this ~4x.")
print("  The plan step is free; only section 6 spends money.")

In [ ]:
# Writes the queue. Refuses to overwrite an existing one.
state.write_shards(shards)

## 4. Smoke test — two shards, one worker

Do not skip. `--max-shards 2` leaves the rest of the queue untouched, and
`--profile` prints where the time went. This is how you find out that the
weight name is wrong or the output prefix is unwritable, for the price of two
shards rather than 40,000.

In [ ]:
smoke = batch.submit_job(
    jobName="western-smoke",
    jobQueue=JOB_QUEUE,
    jobDefinition=JOB_DEF,
    containerOverrides={"command": [
        "work", "--campaign", CAMPAIGN, "--weight", WEIGHT,
        "--procs", "1", "--max-shards", "2", "--profile",
    ]},
)
print("smoke job:", smoke["jobId"])

while True:
    d = batch.describe_jobs(jobs=[smoke["jobId"]])["jobs"][0]
    if d["status"] in ("SUCCEEDED", "FAILED"):
        break
    print(d["status"], end="\r")
    time.sleep(15)
print(f"\n{d['status']}  {d.get('statusReason','')}")
print("log stream:", d.get("container", {}).get("logStreamName"))
print("progress:", state.progress())

In [ ]:
# Look at the picks before committing to the campaign. Numbers that are not
# obviously wrong are not the same as numbers you have looked at.
picks = pd.read_parquet(f"{CAMPAIGN}/picks/")
print(f"{len(picks):,} picks from the smoke test")
print(picks.pha.value_counts().to_dict())
print(f"stations: {sorted(picks.tid.unique())[:6]}")
print(f"conf: {picks.conf.min():.2f}-{picks.conf.max():.2f}, "
      f"amp set on {picks.amp.notna().sum():,}/{len(picks):,}")
picks.head()

## 5. Launch

Only after the smoke test picks have been looked at.

`N_WORKERS x 8` vCPU must stay inside the compute environment's `maxvCpus`
(4000), which binds well before the account quota (12,000). Start conservative:
workers are stateless, so adding more later costs nothing and needs no cleanup.

**Turn the cost alerts on first** — see
[15_monitoring.md](../docs/rerun_2026/15_monitoring.md). A campaign this size
should not run unwatched.

In [ ]:
N_WORKERS = 25          # x 8 vCPU = 200 vCPU; raise once throughput is known
PROCS     = "4"

cap = ce["computeResources"]["maxvCpus"]
asked = N_WORKERS * 8
print(f"requesting {asked} vCPU against maxvCpus {cap}")
assert asked <= cap, "raise maxvCpus on the compute environment, or lower N_WORKERS"

ids = []
for i in range(N_WORKERS):
    r = batch.submit_job(
        jobName=f"western-{i:03d}",
        jobQueue=JOB_QUEUE,
        jobDefinition=JOB_DEF,
        parameters={"campaign": CAMPAIGN, "weight": WEIGHT,
                    "procs": PROCS, "checkpoint": "40"},
    )
    ids.append(r["jobId"])
json.dump(ids, open("western_workers.json", "w"))
print(f"submitted {len(ids)} workers")

## 6. Watch

In [ ]:
p = state.progress()
print(p)
if p["total"]:
    print(f"{100*p['complete']/p['total']:.2f}% complete")

st = {}
for s in ("RUNNING", "RUNNABLE", "SUCCEEDED", "FAILED"):
    st[s] = len(batch.list_jobs(jobQueue=JOB_QUEUE, jobStatus=s)["jobSummaryList"])
print("batch:", st)

## 7. Stopping

Cancelling loses nothing: each shard returns to the queue and resumes from its
last checkpoint, at most 40 station-day-channels back. There is no instance to
terminate — Fargate tasks end with the job.

In [ ]:
# for jid in json.load(open("western_workers.json")):
#     batch.cancel_job(jobId=jid, reason="operator stop")
# print("cancelled; re-run section 5 to resume where the queue left off")